In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import numpy as np
import seaborn as sns
pd.set_option('display.max_rows', None)

In [ ]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
val = pd.read_csv('data/val.csv')

In [ ]:
x_cols = ['Avg source IP count', 'Detect count_y', 'Victim IP_y', 'Port number_y', 
          'Packet speed_y', 'Data speed_y', 'Avg packet len_y', 'Source IP count', 
          'Packet speed_y_normalized', 'Data speed_y_normalized', 'time_of_day',
          'Avg packet len_y_normalized', 'total_seconds', 'weekday_number', 
          'IsWeekend', 'Start Hour', 'Sin_Hour', 'Cos_Hour', 'DayOfYear', 
          'Sin_DayOfYear', 'Cos_DayOfYear', 'Mean_DataSpeed', 'Std_DataSpeed', 
          'Min_DataSpeed', 'Max_DataSpeed', 'Mean_DetectCount', 'Std_DetectCount', 
          'Min_DetectCount', 'Max_DetectCount', 'VictimIP_Count', 'PortNumber_Count', 
          'AvgPacketLen_Mean', 'AvgPacketLen_Std', 'DataSpeed_PacketSpeed', 
          'PortFrequency', 'Std_DataSpeed_Replaced', 'Std_DetectCount_Replaced', 
          'AvgPacketLen_Std_Replaced', 'packet_Total', 'PacketSpeed_Per_Second', 
          'DataSpeed_Per_TotalSeconds', 'AvgPacketLen_Per_TotalSeconds', 
          'PCA_1', 'PCA_2', 'PCA_3', 'PCA_4', 'PCA_5', 'Is_HTTP', 'Is_HTTPS', 
          'Is_FTP_Control', 'Is_FTP_Data', 'Is_SSH', 'Is_Telnet', 'Is_SMTP', 
          'Is_DNS', 'Is_POP3', 'Is_IMAP', 'Is_DHCP', 'Is_SNMP', 'Is_LDAP', 
          'Is_LDAPS', 'Is_SMB_CIFS', 'Is_RDP', 'Is_SIP', 'Is_TFTP', 'Is_MySQL', 
          'Is_PostgreSQL', 'Is_Oracle', 'Is_HTTP_Alt_8080', 'Is_HTTP_Alt_8081', 
          'Is_HTTP_Alt_80', 'Is_HTTPS_Alt_8443', 'Is_Syslog', 'Is_VNC', 'Is_IRC', 
          'Is_NTP', 'Is_Kerberos', 'Is_LDAP_Alt', 'Is_LDAPS_Alt', 'Is_RADIUS', 
          'Is_PPTP', 'Is_RTSP', 'Is_X11', 'Is_SNMP_Trap', 'Is_BGP', 'Is_IMAPS_Alt', 
          'Is_POP3S_Alt', 'Is_Telnet_SSL', 'Is_NNTP', 'Is_NNTPS', 'Is_LDAP_TLS', 
          'Is_AFS', 'Is_NFS', 'Is_SOCKS', 'Is_RSYNC', 'Is_CUPS', 'Is_TFTP_Alt', 
          'Is_Modbus', 'Is_CoAP', 'Is_MQTT', 'Is_AMQP', 'Is_Redis', 'Is_Memcached', 
          'Is_Elasticsearch', 'Is_Zookeeper', 'Is_Cassandra', 'Is_Docker', 
          'Is_Kubernetes', 'Is_SMB_Direct', 'Is_iSCSI', 'Is_AFP', 'Is_DHCPv6', 
          'Is_RIPng', 'Is_OSPF', 'Is_PPPoE', 'Is_L2TP', 'Is_GRE', 'Is_ESP', 'Is_AH']

In [ ]:
bools = ['Is_HTTP', 'Is_HTTPS', 'Is_FTP_Control','Is_FTP_Data','Is_SSH',
       'Is_Telnet','Is_SMTP','Is_DNS','Is_POP3','Is_IMAP','Is_DHCP','Is_SNMP',
       'Is_LDAP','Is_LDAPS','Is_SMB_CIFS','Is_RDP','Is_SIP','Is_TFTP','Is_MySQL',
       'Is_PostgreSQL','Is_Oracle','Is_HTTP_Alt_8080','Is_HTTP_Alt_8081',
       'Is_HTTP_Alt_80','Is_HTTPS_Alt_8443','Is_Syslog','Is_VNC','Is_IRC', 
       'Is_NTP','Is_Kerberos','Is_LDAP_Alt','Is_LDAPS_Alt','Is_RADIUS',
       'Is_PPTP','Is_RTSP','Is_X11','Is_SNMP_Trap','Is_BGP','Is_IMAPS_Alt',
       'Is_POP3S_Alt','Is_Telnet_SSL','Is_NNTP','Is_NNTPS','Is_LDAP_TLS',
       'Is_AFS','Is_NFS','Is_SOCKS','Is_RSYNC','Is_CUPS','Is_TFTP_Alt',
       'Is_Modbus','Is_CoAP','Is_MQTT','Is_AMQP','Is_Redis','Is_Memcached',
       'Is_Elasticsearch','Is_Zookeeper','Is_Cassandra','Is_Docker',
       'Is_Kubernetes','Is_SMB_Direct','Is_iSCSI','Is_AFP','Is_DHCPv6',
       'Is_RIPng','Is_OSPF','Is_PPPoE','Is_L2TP', 'Is_GRE','Is_ESP','Is_AH']
for col in bools:
    if train[col].dtype == 'bool':
        train[col] = train[col].map({True: 1, False: 0})
    if test[col].dtype == 'bool':
        test[col] = test[col].map({True: 1, False: 0})
    if val[col].dtype == 'bool':
        val[col] = val[col].map({True: 1, False: 0})

In [ ]:
from keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='recall', patience=5, restore_best_weights=True)

In [ ]:
from tensorflow.keras.utils import to_categorical
y_train_encoded = to_categorical(train['Type'], num_classes=3)
y_test_encoded = to_categorical(test['Type'], num_classes=3)

In [ ]:
X_train_values = train[x_cols].values.astype('float32')
X_train_reshaped = np.expand_dims(train[x_cols].values, axis=1)
X_test_reshaped = np.expand_dims(test[x_cols].values, axis=1)

# CNN


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
model = keras.Sequential([
    layers.Input(shape=(118,1)), # Input layer now expects (num_features,)
    layers.Flatten(),
    layers.Dense(units=256, activation='relu'),
    layers.Dense(units=128, activation='relu'),
    layers.Dense(units=3, activation='softmax')
])
model.summary()

In [ ]:
model.compile(optimizer='adam',loss='categorical_crossentropy', metrics=['accuracy', 'precision', 'recall'])

In [ ]:
with tf.device('/GPU:0'):
   model.fit(X_train_reshaped,y_train_encoded,batch_size=32,epochs=200,verbose=1,callbacks=[early_stopping])

In [ ]:
loss, accuracy, precision, recall = model.evaluate(test[x_cols], y_test_encoded, verbose=1)

print(f"Test Loss:     {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall:    {recall:.4f}")

In [ ]:
#wo smote
"""Test Loss:     0.6736
Test Accuracy: 0.7370"""
#w smote
"""Test Loss:     0.6572
Test Accuracy: 0.7370"""

# Simpler CNN


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

input_shape = (len(x_cols), 1)

simple_model = keras.Sequential([
    layers.Input(shape=input_shape),
    layers.Flatten(),
    layers.Dense(units=3, activation='softmax')
])
simple_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy', 'precision', 'recall'])

with tf.device('/GPU:0'):
    try:
        simple_model.fit(X_train_reshaped, y_train_encoded, batch_size=32, epochs=200, verbose=1,callbacks=[early_stopping])
    except ValueError as e:
        print(f"ValueError with simple model: {e}")

Epoch 1/200
68138/68138 ━━━━━━━━━━━━━━━━━━━━ 79s 1ms/step - accuracy: 0.6700 - loss: 1167422.7500 - precision: 0.6700 - recall: 0.6700
Epoch 2/200
68138/68138 ━━━━━━━━━━━━━━━━━━━━ 75s 1ms/step - accuracy: 0.7047 - loss: 19196.7480 - precision: 0.7047 - recall: 0.7047
Epoch 3/200
68138/68138 ━━━━━━━━━━━━━━━━━━━━ 75s 1ms/step - accuracy: 0.7121 - loss: 18142.2207 - precision: 0.7121 - recall: 0.7121
Epoch 4/200
68138/68138 ━━━━━━━━━━━━━━━━━━━━ 75s 1ms/step - accuracy: 0.7134 - loss: 21783.2559 - precision: 0.7134 - recall: 0.7134
Epoch 5/200
68138/68138 ━━━━━━━━━━━━━━━━━━━━ 74s 1ms/step - accuracy: 0.7142 - loss: 20995.9180 - precision: 0.7142 - recall: 0.7142
Epoch 6/200
68138/68138 ━━━━━━━━━━━━━━━━━━━━ 74s 1ms/step - accuracy: 0.7178 - loss: 20770.4395 - precision: 0.7178 - recall: 0.7178
Epoch 7/200
68138/68138 ━━━━━━━━━━━━━━━━━━━━ 76s 1ms/step - accuracy: 0.7224 - loss: 16279.9854 - precision: 0.7224 - recall: 0.7224
Epoch 8/200
68138/68138 ━━━━━━━━━━━━━━━━━━━━ 75s 1ms/step - accurac

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
# Predict the labels for the test data
test['pred'] = np.argmax(simple_model.predict(X_test_reshaped), axis=1)
# Example metrics calculation
accuracy = accuracy_score(test['Type'], test['pred'])
print("Accuracy:", accuracy)
precision = precision_score(test['Type'], test['pred'], average='macro')
print("Precision:", precision)
recall = recall_score(test['Type'], test['pred'], average='macro')
print("Recall:", recall)
f1 = f1_score(test['Type'], test['pred'], average='macro')
print("F1-Score:", f1)

38978/38978 ━━━━━━━━━━━━━━━━━━━━ 27s 685us/step
Accuracy: 0.8107115883861181
Precision: 0.5431058036742651
Recall: 0.49998857929690566
F1-Score: 0.5149657706658115


In [ ]:
#wo smote
"""Test Loss:     11098.7607
Test Accuracy: 0.7891"""
#w smote
"""Test Loss:     10123.3623
Test Accuracy: 0.7939"""

# Transformer


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer
import numpy as np

class PositionalEncoding(Layer):
    def __init__(self, sequence_length, d_model):
        super().__init__()
        self.pos_encoding = self.positional_encoding(sequence_length, d_model)

    def get_config(self):
        config = super().get_config()
        return config

    def positional_encoding(self, pos, d_model):
        angles = np.arange(pos)[:, np.newaxis] / np.power(10000, (2 * (np.arange(d_model)[np.newaxis, :] // 2)) / np.float32(d_model))
        angles[:, 0::2] = np.sin(angles[:, 0::2])
        angles[:, 1::2] = np.cos(angles[:, 1::2])
        return tf.cast(angles[np.newaxis, ...], dtype=tf.float32)

    def call(self, x):
        return x + self.pos_encoding[:, :tf.shape(x)[1], :]

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D

def transformer_block(inputs, num_heads, ff_dim, dropout=0.1):
    attention = MultiHeadAttention(num_heads=num_heads, key_dim=inputs.shape[-1])(inputs, inputs)
    attention = Dropout(dropout)(attention)
    out1 = LayerNormalization(epsilon=1e-6)(inputs + attention)

    ffn = Dense(ff_dim, activation='relu')(out1)
    ffn = Dense(inputs.shape[-1])(ffn)
    ffn = Dropout(dropout)(ffn)
    return LayerNormalization(epsilon=1e-6)(out1 + ffn)

def build_transformer_model(sequence_len, num_features, num_heads=4, ff_dim=64):
    inputs = Input(shape=(sequence_len, num_features))
    x = PositionalEncoding(sequence_len, num_features)(inputs)
    x = transformer_block(x, num_heads, ff_dim)
    x = transformer_block(x, num_heads, ff_dim)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.2)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.1)(x)
    outputs = Dense(1, activation='sigmoid')(x)
    return Model(inputs=inputs, outputs=outputs)

In [ ]:
model = build_transformer_model(sequence_len=1, num_features=len(x_cols))
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy', 'precision', 'recall'])
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 1, 46)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_encodin… │ (None, 1, 46)     │          0 │ input_layer_1[0]… │
│ (PositionalEncodin… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 1, 46)     │     34,454 │ positional_encod… │
│ (MultiHeadAttentio… │                   │            │ positional_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 1, 46)     │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 1, 46)     │          0 │ positional_encod… │
│                     │                   │            │ dropout_9[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1, 46)     │         92 │ add_4[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 1, 64)     │      3,008 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 1, 46)     │      2,990 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 1, 46)     │          0 │ dense_7[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_5 (Add)         │ (None, 1, 46)     │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_10[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1, 46)     │         92 │ add_5[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 1, 46)     │     34,454 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_12          │ (None, 1, 46)     │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_6 (Add)         │ (None, 1, 46)     │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_12[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1, 46)     │         92 │ add_6[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 1, 64)     │      3,008 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 1, 46)     │      2,990 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_13          │ (None, 1, 46)     │          0 │ dense_9[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 84,345 (329.47 KB)

 Trainable params: 84,345 (329.47 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
X_train_reshaped = np.expand_dims(train[x_cols].values, axis=1)

In [ ]:
with tf.device('/GPU:0'):
    model.fit(X_train_reshaped,train['Type'],batch_size=32,epochs=200,verbose=1,callbacks=[early_stopping])

Epoch 1/200


/mnt/c/Users/Admin/myenv/lib/python3.12/site-packages/keras/src/losses/losses.py:33: SyntaxWarning: In loss categorical_crossentropy, expected y_pred.shape to be (batch_size, num_classes) with num_classes > 1. Received: y_pred.shape=(None, 1). Consider using 'binary_crossentropy' if you only have 2 classes.
  return self.fn(y_true, y_pred, **self._fn_kwargs)
I0000 00:00:1744715736.496762    3358 service.cc:148] XLA service 0x7f9d54004220 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1744715736.497314    3358 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce GTX 1660 Ti, Compute Capability 7.5
2025-04-15 13:15:36.878317: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1744715737.601298    3358 cuda_dnn.cc:529] Loaded cuDNN version 90300


   31/56878 ━━━━━━━━━━━━━━━━━━━━ 4:55 5ms/step - accuracy: 0.2528 - loss: 0.0000e+00 - precision: 0.8373 - recall: 0.1532    

I0000 00:00:1744715742.047355    3358 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


56878/56878 ━━━━━━━━━━━━━━━━━━━━ 315s 5ms/step - accuracy: 0.1545 - loss: 0.0000e+00 - precision: 0.8333 - recall: 2.5417e-04
Epoch 2/200
56878/56878 ━━━━━━━━━━━━━━━━━━━━ 297s 5ms/step - accuracy: 0.1540 - loss: 0.0000e+00 - precision: 0.0000e+00 - recall: 0.0000e+00
Epoch 3/200
56878/56878 ━━━━━━━━━━━━━━━━━━━━ 309s 5ms/step - accuracy: 0.1544 - loss: 0.0000e+00 - precision: 0.0000e+00 - recall: 0.0000e+00
Epoch 4/200
56878/56878 ━━━━━━━━━━━━━━━━━━━━ 304s 5ms/step - accuracy: 0.1544 - loss: 0.0000e+00 - precision: 0.0000e+00 - recall: 0.0000e+00
Epoch 5/200
56878/56878 ━━━━━━━━━━━━━━━━━━━━ 304s 5ms/step - accuracy: 0.1538 - loss: nan - precision: 0.0000e+00 - recall: 0.0000e+00       
Epoch 6/200
56878/56878 ━━━━━━━━━━━━━━━━━━━━ 297s 5ms/step - accuracy: 0.1544 - loss: nan - precision: 0.0000e+00 - recall: 0.0000e+00


In [ ]:
X_test_reshaped = np.expand_dims(test[x_cols].values, axis=1)

In [ ]:
# Predict the labels for the test data
test['pred'] = model.predict(X_test_reshaped)

# Example metrics calculation
accuracy = accuracy_score(test['Type'], test['pred'])
print("Accuracy:", accuracy)
precision = precision_score(test['Type'], test['pred'], average='macro')
print("Precision:", precision)
recall = recall_score(test['Type'], test['pred'], average='macro')
print("Recall:", recall)
f1 = f1_score(test['Type'], test['pred'], average='macro')
print("F1-Score:", f1)

38978/38978 ━━━━━━━━━━━━━━━━━━━━ 70s 2ms/step 
Accuracy: 0.24466873946696213
Precision: 0.08155624648898738
Recall: 0.3333333333333333
F1-Score: 0.1310489191267299


/mnt/c/Users/Admin/myenv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
#wo smote
"""Accuracy: 0.24466873946696213
Precision: 0.08155624648898738
Recall: 0.3333333333333333
F1-Score: 0.1310489191267299"""
#w smote
""""""

# Pre-trained models
